# 04 Trading Paper Recommendations

Reads anomaly outputs and immutable index cards, filters eligible strong anomalies, enforces anti-ex-post-rationalization timing, and writes paper-only recommended trades to Excel. It never places real trades.


## Setup


In [1]:
from __future__ import annotations

import hashlib
import json
import math
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Optional

import numpy as np
import pandas as pd
import yaml

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 180)


def find_project_root(start: Path | None = None) -> Path:
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "config" / "falnama_config.yaml").exists() or (candidate / "polymarket_geopolitics_anomaly_detection_pilot.ipynb").exists():
            return candidate
    raise RuntimeError("Could not locate Falnama project root. Run from inside the Falnama folder.")

PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "config" / "falnama_config.yaml"
RUN_TIME_UTC = datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace("+00:00", "Z")


def load_config() -> dict[str, Any]:
    with CONFIG_PATH.open("r", encoding="utf-8") as f:
        return yaml.safe_load(f) or {}

CONFIG = load_config()
REPOSITORIES = CONFIG.get("repositories", {})
for repo_rel in REPOSITORIES.values():
    (PROJECT_ROOT / repo_rel).mkdir(parents=True, exist_ok=True)

RUN_LOG_DIR = PROJECT_ROOT / REPOSITORIES.get("run_logs", "repositories/run_logs")
RUN_LOG_DIR.mkdir(parents=True, exist_ok=True)


def repo_path(key: str, default: str) -> Path:
    path = PROJECT_ROOT / REPOSITORIES.get(key, default)
    path.mkdir(parents=True, exist_ok=True)
    return path


def write_run_log(notebook_name: str, records: list[dict[str, Any]]) -> Path:
    path = RUN_LOG_DIR / f"{notebook_name}_{RUN_TIME_UTC.replace(':', '').replace('-', '')}.jsonl"
    with path.open("x", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps({"run_time_utc": RUN_TIME_UTC, **record}, default=str) + "\n")
    return path


def load_csv_nonempty(path: Path) -> pd.DataFrame | None:
    if not path.exists() or path.stat().st_size <= 1:
        return None
    try:
        df = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        return None
    return df if not df.empty else None


def normalize_timestamp(value: Any) -> pd.Timestamp:
    if value is None or value == "" or (isinstance(value, float) and np.isnan(value)):
        return pd.NaT
    if isinstance(value, pd.Timestamp):
        return value.tz_localize("UTC") if value.tzinfo is None else value.tz_convert("UTC")
    if isinstance(value, (int, float, np.integer, np.floating)):
        unit = "ms" if float(value) > 10_000_000_000 else "s"
        return pd.to_datetime(value, unit=unit, utc=True, errors="coerce")
    return pd.to_datetime(value, utc=True, errors="coerce")


def parse_jsonish(value: Any, default: Any = None) -> Any:
    if default is None:
        default = []
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return default
    if isinstance(value, (list, dict)):
        return value
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return default
        try:
            return json.loads(text)
        except Exception:
            return value
    return value


def first_present(obj: dict[str, Any] | pd.Series, keys: list[str], default: Any = None) -> Any:
    for key in keys:
        if key in obj and obj[key] not in (None, "") and not (isinstance(obj[key], float) and np.isnan(obj[key])):
            return obj[key]
    return default


def safe_slug(value: Any, fallback: str = "unknown") -> str:
    text = str(value if value not in (None, "") else fallback).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "-", text).strip("-")
    return text[:100] or fallback


## Paper-Only Trading Functions
This cell implements anomaly eligibility, card matching, immutable-card timing checks, prediction-level rejection reasons, configurable position sizing, and a hard broker-call guardrail.


In [2]:
from jsonschema import Draft202012Validator

RELEVANT_DIR = repo_path("relevant_markets", "repositories/relevant_markets")
INDEX_CARD_DIR = repo_path("index_cards", "repositories/index_cards")
TRADES_DIR = repo_path("recommended_trades", "repositories/recommended_trades")
REJECTED_DIR = repo_path("rejected_signals", "repositories/rejected_signals")
TRADE_SCHEMA_PATH = PROJECT_ROOT / "schemas" / "recommended_trade_schema.json"
CARD_SCHEMA_PATH = PROJECT_ROOT / "schemas" / "index_card_schema.json"
LOGS: list[dict[str, Any]] = []

with TRADE_SCHEMA_PATH.open("r", encoding="utf-8") as f:
    TRADE_VALIDATOR = Draft202012Validator(json.load(f))
with CARD_SCHEMA_PATH.open("r", encoding="utf-8") as f:
    CARD_VALIDATOR = Draft202012Validator(json.load(f))


def assert_no_real_broker_execution() -> None:
    if not CONFIG.get("paper_trading_only", True) or CONFIG.get("allow_real_broker_execution", False):
        raise RuntimeError("Real broker execution is disabled in this paper-only notebook.")


def call_real_broker_api(*args: Any, **kwargs: Any) -> None:
    raise RuntimeError("Hard guardrail: real broker API calls are forbidden in Falnama paper recommendations.")

assert_no_real_broker_execution()


def newest_files(directory: Path, pattern: str) -> list[Path]:
    return sorted(directory.glob(pattern), key=lambda p: p.stat().st_mtime, reverse=True)


def pick_column(df: pd.DataFrame, names: list[str]) -> str | None:
    lower = {c.lower(): c for c in df.columns}
    for name in names:
        if name.lower() in lower:
            return lower[name.lower()]
    return None


def load_anomalies() -> tuple[pd.DataFrame, str | None]:
    # Prefer nonempty strong anomaly files, including the original pilot output.
    for path in newest_files(RELEVANT_DIR, "strong_anomalies_*.csv"):
        df = load_csv_nonempty(path)
        if df is not None:
            return df, str(path)
    legacy_strong = load_csv_nonempty(PROJECT_ROOT / CONFIG.get("legacy_inputs", {}).get("legacy_strong_anomalies", ""))
    if legacy_strong is not None:
        return legacy_strong, str(PROJECT_ROOT / CONFIG.get("legacy_inputs", {}).get("legacy_strong_anomalies", ""))
    for path in newest_files(RELEVANT_DIR, "ranked_anomalies_*.csv"):
        df = load_csv_nonempty(path)
        if df is not None:
            return df, str(path)
    legacy_ranked = load_csv_nonempty(PROJECT_ROOT / CONFIG.get("legacy_inputs", {}).get("legacy_ranked_anomalies", ""))
    if legacy_ranked is not None:
        return legacy_ranked, str(PROJECT_ROOT / CONFIG.get("legacy_inputs", {}).get("legacy_ranked_anomalies", ""))
    if CONFIG.get("anomaly_detector", {}).get("use_mock_if_no_inputs", True):
        return pd.DataFrame([{"rank": 1, "overall_anomaly_score": 92.0, "anomaly_class": "strong", "market_id": "mock-market-001", "market_slug": "mock-geopolitical-risk-market", "market_name": "Mock geopolitical escalation market for smoke testing", "trigger_time_utc": RUN_TIME_UTC}]), None
    return pd.DataFrame(), None

def load_index_cards() -> list[dict[str, Any]]:
    cards = []
    for path in sorted(INDEX_CARD_DIR.glob("*.json")):
        try:
            card = json.loads(path.read_text(encoding="utf-8"))
            errors = sorted(CARD_VALIDATOR.iter_errors(card), key=lambda e: e.path)
            if errors:
                LOGS.append({"event": "card_schema_validation_failure", "path": str(path), "errors": [e.message for e in errors]})
                continue
            card["_path"] = str(path)
            cards.append(card)
        except Exception as exc:
            LOGS.append({"event": "card_load_failure", "path": str(path), "error": repr(exc)})
    return cards


def normalize_key(value: Any) -> str | None:
    if value is None or value == "" or (isinstance(value, float) and np.isnan(value)):
        return None
    return str(value).strip().lower()


def build_card_indexes(cards: list[dict[str, Any]]) -> dict[str, dict[str, list[dict[str, Any]]]]:
    indexes = {"market_id": {}, "market_slug": {}, "market_name": {}}
    for card in cards:
        source = card.get("source", {})
        for key in indexes:
            value = normalize_key(source.get(key) or (card.get("market_name") if key == "market_name" else None))
            if value:
                indexes[key].setdefault(value, []).append(card)
    for by_key in indexes.values():
        for matches in by_key.values():
            matches.sort(key=lambda c: c.get("created_time_utc", ""), reverse=True)
    return indexes


def match_cards(row: pd.Series, indexes: dict[str, dict[str, list[dict[str, Any]]]]) -> list[dict[str, Any]]:
    probes = {
        "market_id": first_present(row, ["market_id", "id"], None),
        "market_slug": first_present(row, ["market_slug", "slug"], None),
        "market_name": first_present(row, ["market_name", "question", "title"], None),
    }
    for key, value in probes.items():
        normalized = normalize_key(value)
        if normalized and normalized in indexes[key]:
            return indexes[key][normalized]
    return []


def basic_rejection_from_row(row: pd.Series, reason: str) -> dict[str, Any]:
    return {
        "run_time_utc": RUN_TIME_UTC,
        "market_id": first_present(row, ["market_id", "id"], None),
        "market_slug": first_present(row, ["market_slug", "slug"], None),
        "market_name": first_present(row, ["market_name", "question", "title"], None),
        "card_id": None,
        "ticker": None,
        "reason": reason,
    }


def eligible_anomalies(df: pd.DataFrame) -> tuple[pd.DataFrame, list[dict[str, Any]], dict[str, str | None]]:
    rejected: list[dict[str, Any]] = []
    cols = {
        "class": pick_column(df, ["anomaly_class", "class"]),
        "rank": pick_column(df, ["rank", "anomaly_rank"]),
        "score": pick_column(df, ["anomaly_score", "composite_score", "overall_anomaly_score"]),
        "market_id": pick_column(df, ["market_id", "id"]),
        "market_slug": pick_column(df, ["market_slug", "slug"]),
        "market_name": pick_column(df, ["market_name", "question", "title"]),
        "trigger": pick_column(df, ["trigger_time_utc", "anomaly_timestamp", "timestamp_utc"]),
    }
    if cols["class"] is None:
        for _, row in df.iterrows():
            rejected.append(basic_rejection_from_row(row, "missing anomaly_class column"))
        return pd.DataFrame(), rejected, cols
    strong_label = str(CONFIG.get("eligible_anomaly_class", "strong")).lower()
    strong = df[df[cols["class"]].astype(str).str.lower() == strong_label].copy()
    nonstrong = df[df[cols["class"]].astype(str).str.lower() != strong_label]
    for _, row in nonstrong.iterrows():
        rejected.append(basic_rejection_from_row(row, "anomaly_class is not eligible strong class"))
    if strong.empty:
        return strong, rejected, cols
    if cols["rank"]:
        strong["_eligibility_sort"] = pd.to_numeric(strong[cols["rank"]], errors="coerce")
        strong = strong.sort_values("_eligibility_sort", ascending=True, na_position="last")
    elif cols["score"]:
        strong["_eligibility_sort"] = pd.to_numeric(strong[cols["score"]], errors="coerce")
        strong = strong.sort_values("_eligibility_sort", ascending=False, na_position="last")
    else:
        for _, row in strong.iterrows():
            rejected.append(basic_rejection_from_row(row, "missing rank and score columns"))
        return pd.DataFrame(), rejected, cols
    cutoff = float(CONFIG.get("strong_rank_percentile_cutoff", 0.50))
    keep_n = max(1, math.ceil(len(strong) * cutoff))
    eligible = strong.head(keep_n).copy()
    ineligible = strong.iloc[keep_n:]
    for _, row in ineligible.iterrows():
        rejected.append(basic_rejection_from_row(row, "anomaly not in top half of strong class"))
    if cols["score"]:
        scores = pd.to_numeric(eligible[cols["score"]], errors="coerce")
        eligible["_normalized_anomaly_strength"] = (scores / float(CONFIG.get("position_score_caps", {}).get("anomaly_strength_score", 100))).clip(0, 1).fillna(0)
    else:
        eligible["_normalized_anomaly_strength"] = np.linspace(1.0, 0.5, len(eligible)) if len(eligible) > 1 else 1.0
    return eligible, rejected, cols


def card_existed_before_trigger(card: dict[str, Any], trigger: Any) -> bool:
    if CONFIG.get("backfill_testing_mode", False):
        return True
    trigger_ts = normalize_timestamp(trigger)
    card_ts = normalize_timestamp(card.get("created_time_utc"))
    if pd.isna(trigger_ts) or pd.isna(card_ts):
        return False
    return card_ts <= trigger_ts


def compute_position(confidence: float, expected_bps: float, anomaly_strength: float) -> tuple[float, float, float, float, float]:
    caps = CONFIG.get("position_score_caps", {})
    weights = CONFIG.get("position_score_weights", {})
    norm_conf = float(np.clip(confidence / float(caps.get("confidence", 1.0)), 0, 1))
    norm_mag = float(np.clip(abs(expected_bps) / float(caps.get("expected_magnitude_bps", 2000)), 0, 1))
    norm_anom = float(np.clip(anomaly_strength, 0, 1))
    score = (norm_conf ** float(weights.get("confidence", 1.0))) * (norm_mag ** float(weights.get("expected_magnitude", 1.0))) * (norm_anom ** float(weights.get("anomaly_strength", 1.0)))
    score = float(np.clip(score, 0, 1))
    notional = round(float(CONFIG.get("max_position_usd", 10000)) * score, 2)
    return norm_conf, norm_mag, norm_anom, score, notional


def make_rejection(row: pd.Series | dict[str, Any], reason: str, prediction: dict[str, Any] | None = None, card: dict[str, Any] | None = None) -> dict[str, Any]:
    getter = row.get if hasattr(row, "get") else lambda k, d=None: d
    return {
        "run_time_utc": RUN_TIME_UTC,
        "market_id": getter("market_id", None),
        "market_slug": getter("market_slug", None),
        "market_name": getter("market_name", getter("question", None)),
        "card_id": card.get("card_id") if card else None,
        "ticker": prediction.get("ticker") if prediction else None,
        "reason": reason,
    }


## Run Paper Recommendation Engine


In [3]:
anomalies_df, anomalies_source = load_anomalies()
cards = load_index_cards()
indexes = build_card_indexes(cards)
eligible_df, rejected, cols = eligible_anomalies(anomalies_df)
recommended: list[dict[str, Any]] = []

for _, row in eligible_df.iterrows():
    cards_for_row = match_cards(row, indexes)
    if not cards_for_row:
        rejected.append(make_rejection(row, "no matching index card"))
        continue
    trigger_value = first_present(row, [cols.get("trigger") or "", "trigger_time_utc", "anomaly_timestamp", "timestamp_utc"], None)
    valid_time_cards = [card for card in cards_for_row if card_existed_before_trigger(card, trigger_value)]
    if not valid_time_cards:
        rejected.append(make_rejection(row, "card created after trigger time", card=cards_for_row[0]))
        continue
    card = valid_time_cards[0]
    anomaly_strength = float(row.get("_normalized_anomaly_strength", 0))
    for prediction in card.get("predictions", []):
        expected = prediction.get("expected_return_12h_bps")
        confidence = prediction.get("confidence")
        direction = str(prediction.get("expected_direction", "")).lower()
        ticker = prediction.get("ticker")
        instrument = prediction.get("tradable_instrument_name")
        if expected is None or abs(float(expected)) < float(CONFIG.get("minimum_expected_move_bps", 700)):
            rejected.append(make_rejection(row, "expected move below threshold", prediction, card))
            continue
        if not ticker and not instrument:
            rejected.append(make_rejection(row, "missing ticker/tradable instrument", prediction, card))
            continue
        if confidence is None or not (0 <= float(confidence) <= 1):
            rejected.append(make_rejection(row, "missing confidence", prediction, card))
            continue
        if direction not in {"up", "down"}:
            rejected.append(make_rejection(row, "invalid direction", prediction, card))
            continue
        norm_conf, norm_mag, norm_anom, score, notional = compute_position(float(confidence), float(expected), anomaly_strength)
        ci = prediction.get("confidence_interval_bps") or [None, None]
        trade = {
            "run_time_utc": RUN_TIME_UTC,
            "trigger_time_utc": None if pd.isna(normalize_timestamp(trigger_value)) else normalize_timestamp(trigger_value).isoformat().replace("+00:00", "Z"),
            "market_id": first_present(row, ["market_id", "id"], None),
            "market_slug": first_present(row, ["market_slug", "slug"], None),
            "market_name": first_present(row, ["market_name", "question", "title"], None),
            "anomaly_class": first_present(row, [cols.get("class") or "", "anomaly_class"], None),
            "anomaly_rank_or_score": first_present(row, [cols.get("rank") or "", cols.get("score") or "", "rank", "overall_anomaly_score"], None),
            "card_id": card["card_id"],
            "card_created_time_utc": card["created_time_utc"],
            "card_hash": card["card_hash"],
            "asset": prediction.get("asset"),
            "ticker": ticker,
            "tradable_instrument_name": instrument,
            "asset_class": prediction.get("asset_class"),
            "expected_direction": direction,
            "expected_return_12h_bps": float(expected),
            "confidence": float(confidence),
            "confidence_interval_lower_bps": ci[0] if len(ci) > 0 else None,
            "confidence_interval_upper_bps": ci[1] if len(ci) > 1 else None,
            "normalized_confidence": norm_conf,
            "normalized_expected_magnitude": norm_mag,
            "normalized_anomaly_strength": norm_anom,
            "position_score": score,
            "recommended_notional_usd": notional,
            "paper_trade_action": "PAPER_BUY_OR_LONG" if direction == "up" else "PAPER_SELL_OR_SHORT",
            "reasoning_summary": prediction.get("reasoning"),
            "uncertainty_notes": " | ".join(card.get("uncertainty_notes", [])),
            "time_plan_30m": prediction.get("time_plan", {}).get("30m"),
            "time_plan_1h": prediction.get("time_plan", {}).get("1h"),
            "time_plan_2h": prediction.get("time_plan", {}).get("2h"),
            "time_plan_6h": prediction.get("time_plan", {}).get("6h"),
            "time_plan_12h": prediction.get("time_plan", {}).get("12h"),
            "output_mode": "BACKTEST_NON_LIVE" if CONFIG.get("backfill_testing_mode", False) else "PAPER_ONLY_LIVE_GUARDED",
        }
        errors = sorted(TRADE_VALIDATOR.iter_errors(trade), key=lambda e: e.path)
        if errors:
            rejected.append(make_rejection(row, "schema validation failure: " + "; ".join(e.message for e in errors), prediction, card))
            continue
        recommended.append(trade)

RECOMMENDED_TRADE_COLUMNS = [
    "run_time_utc", "trigger_time_utc", "market_id", "market_slug", "market_name", "anomaly_class",
    "anomaly_rank_or_score", "card_id", "card_created_time_utc", "card_hash", "asset", "ticker",
    "tradable_instrument_name", "asset_class", "expected_direction", "expected_return_12h_bps", "confidence",
    "confidence_interval_lower_bps", "confidence_interval_upper_bps", "normalized_confidence",
    "normalized_expected_magnitude", "normalized_anomaly_strength", "position_score", "recommended_notional_usd",
    "paper_trade_action", "reasoning_summary", "uncertainty_notes", "time_plan_30m", "time_plan_1h",
    "time_plan_2h", "time_plan_6h", "time_plan_12h", "output_mode"
]
REJECTED_SIGNAL_COLUMNS = ["run_time_utc", "market_id", "market_slug", "market_name", "card_id", "ticker", "reason"]
recommended_df = pd.DataFrame(recommended, columns=RECOMMENDED_TRADE_COLUMNS)
rejected_df = pd.DataFrame(rejected)
for col in REJECTED_SIGNAL_COLUMNS:
    if col not in rejected_df.columns:
        rejected_df[col] = None
rejected_df = rejected_df[REJECTED_SIGNAL_COLUMNS]
card_links_df = pd.DataFrame([{"card_id": c.get("card_id"), "card_hash": c.get("card_hash"), "created_time_utc": c.get("created_time_utc"), "market_id": c.get("source", {}).get("market_id"), "market_slug": c.get("source", {}).get("market_slug"), "market_name": c.get("market_name"), "path": c.get("_path")} for c in cards])
config_snapshot_df = pd.json_normalize(CONFIG, sep=".").T.reset_index()
config_snapshot_df.columns = ["config_key", "config_value"]
run_log_df = pd.DataFrame(LOGS + [{"event": "trading_outputs", "anomalies_source": anomalies_source, "input_anomalies": len(anomalies_df), "eligible_anomalies": len(eligible_df), "cards_loaded": len(cards), "recommended_trades": len(recommended_df), "rejected_signals": len(rejected_df)}])

stamp = RUN_TIME_UTC.replace(":", "").replace("-", "")
excel_path = TRADES_DIR / f"paper_recommended_trades_{stamp}.xlsx"
rejected_path = REJECTED_DIR / f"rejected_signals_{stamp}.csv"
rejected_df.to_csv(rejected_path, index=False)
with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    recommended_df.to_excel(writer, sheet_name="Recommended Trades", index=False)
    rejected_df.to_excel(writer, sheet_name="Rejected Signals", index=False)
    card_links_df.to_excel(writer, sheet_name="Index Card Links", index=False)
    config_snapshot_df.to_excel(writer, sheet_name="Config Snapshot", index=False)
    run_log_df.to_excel(writer, sheet_name="Run Log", index=False)
log_path = write_run_log("04_trading_paper_recommendations", run_log_df.to_dict(orient="records"))

print(f"Recommended paper trades: {len(recommended_df):,}")
print(f"Rejected signals: {len(rejected_df):,}")
print(excel_path)
print(rejected_path)
print(log_path)
display(recommended_df.head(20))
display(rejected_df.head(20))


Recommended paper trades: 0
Rejected signals: 24
/Users/R2-D2/Documents/Codex/Falnama/repositories/recommended_trades/paper_recommended_trades_20260609T021539Z.xlsx
/Users/R2-D2/Documents/Codex/Falnama/repositories/rejected_signals/rejected_signals_20260609T021539Z.csv
/Users/R2-D2/Documents/Codex/Falnama/repositories/run_logs/04_trading_paper_recommendations_20260609T021539Z.jsonl


,run_time_utc,trigger_time_utc,market_id,market_slug,market_name,anomaly_class,anomaly_rank_or_score,card_id,card_created_time_utc,card_hash,asset,ticker,tradable_instrument_name,asset_class,expected_direction,expected_return_12h_bps,confidence,confidence_interval_lower_bps,confidence_interval_upper_bps,normalized_confidence,normalized_expected_magnitude,normalized_anomaly_strength,position_score,recommended_notional_usd,paper_trade_action,reasoning_summary,uncertainty_notes,time_plan_30m,time_plan_1h,time_plan_2h,time_plan_6h,time_plan_12h,output_mode


,run_time_utc,market_id,market_slug,market_name,card_id,ticker,reason
0,2026-06-09T02:15:39Z,254197,None,Will another man win the 2024 Republican VP nomination?,None,None,anomaly not in top half of strong class
1,2026-06-09T02:15:39Z,254112,None,Will JD Vance win the 2024 Republican VP nomination?,None,None,anomaly not in top half of strong class
2,2026-06-09T02:15:39Z,1272508,None,Will the People’s Party (PPLE) win between 120 and 134 seats in the 2026 Thai legislative election?,None,None,anomaly not in top half of strong class
3,2026-06-09T02:15:39Z,1272508,None,Will the People’s Party (PPLE) win between 120 and 134 seats in the 2026 Thai legislative election?,None,None,anomaly not in top half of strong class
4,2026-06-09T02:15:39Z,254197,None,Will another man win the 2024 Republican VP nomination?,None,None,anomaly not in top half of strong class
5,2026-06-09T02:15:39Z,1272508,None,Will the People’s Party (PPLE) win between 120 and 134 seats in the 2026 Thai legislative election?,None,None,anomaly not in top half of strong class
6,2026-06-09T02:15:39Z,248211,None,[Single Market] Will Glenn Youngkin win the 2024 Republican presidential nomination?,None,None,anomaly not in top half of strong class
7,2026-06-09T02:15:39Z,248211,None,[Single Market] Will Glenn Youngkin win the 2024 Republican presidential nomination?,None,None,anomaly not in top half of strong class
8,2026-06-09T02:15:39Z,248212,None,[Single Market] Will Tim Scott win the 2024 Republican presidential nomination?,None,None,anomaly not in top half of strong class
9,2026-06-09T02:15:39Z,248212,None,[Single Market] Will Tim Scott win the 2024 Republican presidential nomination?,None,None,anomaly not in top half of strong class
